# TP3 — Modelos Generativos Profundos: VAE, β-VAE y comparación con GAN

**Materia:** Aprendizaje Profundo — UNSAM  
**Tema:** Autoencoders variacionales, espacio latente, generación, regularización KL y comparación conceptual con GANs  
**Modalidad:** individual  
**Entrega:** notebook ejecutado + informe dentro del notebook + defensa oral breve

---

## Idea general

En TP1 trabajamos clasificación supervisada con MLP/CNN.  
En TP2 implementamos atención causal y un mini-GPT autoregresivo.  

En este TP vamos a construir un **modelo generativo profundo**: un **Variational Autoencoder (VAE)** convolucional entrenado en imágenes. El objetivo no es sólo reconstruir imágenes, sino aprender un **espacio latente probabilístico** desde el cual podamos **generar nuevas muestras**.

La consigna combina implementación, experimentación y análisis crítico:

1. Implementar un **ConvVAE** en PyTorch.
2. Implementar la **loss VAE**: reconstrucción + divergencia KL.
3. Entrenar un baseline con **KL annealing**.
4. Evaluar reconstrucciones, muestras desde el prior e interpolaciones latentes.
5. Comparar variantes de **β-VAE**.
6. Analizar posterior collapse.
7. Discutir diferencias entre VAE y GAN.
8. Opcional: extender a caras, GAN mínimo o matrices de conectividad cerebral.

---

## Objetivos de aprendizaje

Al finalizar el TP deberías poder:

- Distinguir un autoencoder determinístico de un VAE.
- Explicar la ELBO y sus dos términos principales.
- Implementar el truco de reparametrización.
- Calcular correctamente la KL entre \(q_\phi(z|x)\) y \(p(z)=\mathcal{N}(0,I)\).
- Interpretar el trade-off reconstrucción/generación en β-VAE.
- Detectar indicios de posterior collapse.
- Evaluar representaciones latentes con visualizaciones e interpolaciones.
- Comparar conceptualmente VAE vs GAN.
- Formular conclusiones críticas sin sobreinterpretar los resultados.

---

## Requisitos mínimos para aprobar

- El notebook debe correr de punta a punta.
- Deben pasar los checkpoints automáticos.
- Debe entrenarse al menos un VAE baseline.
- Deben incluirse reconstrucciones, muestras del prior e interpolaciones.
- Debe completarse al menos una comparación experimental de β.
- Debe responderse el informe final con análisis propio.

## Extensiones opcionales

Elegí **una** si querés apuntar a una nota alta:

- Extensión A: entrenar/adaptar el VAE a **Olivetti Faces**.
- Extensión B: implementar un **GAN mínimo** y compararlo con el VAE.
- Extensión C: adaptar el pipeline a **matrices de conectividad cerebral** sintéticas o provistas por la cátedra.


# 0. Setup reproducible


In [ ]:
import os
import math
import random
import warnings
from dataclasses import dataclass
from typing import Tuple, Dict, Optional, List

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader, Subset, TensorDataset
from torchvision import datasets, transforms
from torchvision.utils import make_grid

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Device: {device}")


# 1. Dataset

Usaremos **FashionMNIST** por defecto. Es más interesante que MNIST porque las clases tienen mayor variabilidad visual: remeras, pantalones, zapatillas, botas, bolsos, etc.

Podés cambiar a `MNIST` si tu computadora es lenta o si querés depurar más rápido.

La entrada del VAE será una imagen \(x \in [0,1]^{1 \times 28 \times 28}\).


In [ ]:
DATASET_NAME = "FashionMNIST"   # opciones: "FashionMNIST" o "MNIST"
DATA_DIR = "./data"

# Para que el TP corra razonablemente en CPU. Para experimentos finales podés usar False.
USE_SUBSET = True
N_TRAIN_SUBSET = 12000
N_VAL_SUBSET = 2000

transform = transforms.ToTensor()

if DATASET_NAME == "FashionMNIST":
    train_full = datasets.FashionMNIST(DATA_DIR, train=True, download=True, transform=transform)
    val_full   = datasets.FashionMNIST(DATA_DIR, train=False, download=True, transform=transform)
    CLASSES = train_full.classes
elif DATASET_NAME == "MNIST":
    train_full = datasets.MNIST(DATA_DIR, train=True, download=True, transform=transform)
    val_full   = datasets.MNIST(DATA_DIR, train=False, download=True, transform=transform)
    CLASSES = [str(i) for i in range(10)]
else:
    raise ValueError("DATASET_NAME debe ser 'FashionMNIST' o 'MNIST'")

if USE_SUBSET:
    g = torch.Generator().manual_seed(SEED)
    train_idx = torch.randperm(len(train_full), generator=g)[:N_TRAIN_SUBSET]
    val_idx   = torch.randperm(len(val_full),   generator=g)[:N_VAL_SUBSET]
    train_ds = Subset(train_full, train_idx)
    val_ds   = Subset(val_full, val_idx)
else:
    train_ds, val_ds = train_full, val_full

print(f"Dataset: {DATASET_NAME}")
print(f"Train samples: {len(train_ds)}")
print(f"Val samples:   {len(val_ds)}")
print(f"Classes: {CLASSES}")


In [ ]:
def show_image_grid(x: torch.Tensor, title: str = "", nrow: int = 8, figsize=(8, 4)):
    # Muestra un batch de imágenes en escala de grises.
    x = x.detach().cpu().clamp(0, 1)
    grid = make_grid(x, nrow=nrow, padding=2)
    plt.figure(figsize=figsize)
    plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray")
    plt.axis("off")
    plt.title(title)
    plt.show()

# Muestra inicial
loader_preview = DataLoader(train_ds, batch_size=32, shuffle=True)
x0, y0 = next(iter(loader_preview))
show_image_grid(x0[:32], title=f"Muestras de {DATASET_NAME}", nrow=8, figsize=(8, 4))
print("Shape de batch:", x0.shape)


# 2. Teoría mínima: de Autoencoder a VAE

Un autoencoder determinístico aprende:

\[
z = f_\phi(x), \qquad \hat{x} = g_\theta(z)
\]

Esto sirve para comprimir y reconstruir, pero no garantiza que el espacio latente sea continuo ni muestreable. Si tomamos un punto aleatorio \(z\), el decoder no necesariamente sabe qué hacer.

Un **Variational Autoencoder** aprende una distribución posterior aproximada:

\[
q_\phi(z|x) = \mathcal{N}(\mu_\phi(x), \mathrm{diag}(\sigma_\phi^2(x)))
\]

y regulariza ese posterior para que se parezca a un prior simple:

\[
p(z) = \mathcal{N}(0,I)
\]

La función objetivo viene de maximizar la **ELBO**:

\[
\mathcal{L}_{ELBO}(x) =
\mathbb{E}_{q_\phi(z|x)}[\log p_\theta(x|z)]
-
D_{KL}(q_\phi(z|x) \| p(z))
\]

En implementación de deep learning solemos **minimizar**:

\[
\mathcal{J}(x) =
\underbrace{\mathcal{L}_{rec}(x, \hat{x})}_{\text{reconstrucción}}
+
\beta
\underbrace{D_{KL}(q_\phi(z|x) \| p(z))}_{\text{regularización}}
\]

donde \(\beta=1\) corresponde al VAE estándar y \(\beta \neq 1\) da lugar al **β-VAE**.

---

## Truco de reparametrización

No podemos backpropagar directamente a través de un muestreo:

\[
z \sim \mathcal{N}(\mu, \sigma^2)
\]

Entonces reescribimos:

\[
z = \mu + \sigma \odot \epsilon, \qquad \epsilon \sim \mathcal{N}(0,I)
\]

Ahora el azar queda aislado en \(\epsilon\), y \(z\) es diferenciable respecto de \(\mu\) y \(\sigma\).

---

## KL cerrada para gaussianas diagonales

Para \(q_\phi(z|x)=\mathcal{N}(\mu, \sigma^2)\) y \(p(z)=\mathcal{N}(0,I)\):

\[
D_{KL}(q_\phi(z|x)\|p(z)) =
-\frac{1}{2}
\sum_j
\left(1+\log\sigma_j^2-\mu_j^2-\sigma_j^2\right)
\]


# 3. Configuración del modelo

La arquitectura será un **ConvVAE**:

```text
x: (B, 1, 28, 28)
   ↓ encoder convolucional
h
   ↓ capas lineales
mu, logvar: (B, latent_dim)
   ↓ reparametrización
z: (B, latent_dim)
   ↓ decoder convolucional
x_hat: (B, 1, 28, 28)
```

El encoder comprime. El decoder genera.


In [ ]:
@dataclass
class VAEConfig:
    image_size: int = 28
    in_channels: int = 1
    hidden_channels: Tuple[int, int] = (32, 64)
    latent_dim: int = 16

    beta: float = 1.0
    free_bits: float = 0.0
    kl_anneal_epochs: int = 4

    batch_size: int = 128
    epochs: int = 12 if device != "cpu" else 5
    lr: float = 1e-3
    weight_decay: float = 1e-5
    num_workers: int = 2

cfg = VAEConfig()
cfg


# 4. Implementación del ConvVAE

Completá los `TODO`.

Puntos importantes:

- `encode(x)` debe devolver `mu` y `logvar`.
- `reparameterize(mu, logvar)` debe implementar \(z=\mu+\sigma\epsilon\).
- `decode(z)` debe devolver una imagen en \([0,1]\). Por eso el decoder termina con `Sigmoid`.
- `forward(x)` debe devolver `recon, mu, logvar, z`.


In [ ]:
class ConvVAE(nn.Module):
    def __init__(self, cfg: VAEConfig):
        super().__init__()
        self.cfg = cfg
        c1, c2 = cfg.hidden_channels

        # Encoder: 28x28 -> 14x14 -> 7x7
        self.encoder = nn.Sequential(
            nn.Conv2d(cfg.in_channels, c1, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(c1),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(c1, c2, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(c2),
            nn.LeakyReLU(0.2, inplace=True),
        )

        # Calculamos dimensionalidad automáticamente para evitar hardcodeos.
        with torch.no_grad():
            dummy = torch.zeros(1, cfg.in_channels, cfg.image_size, cfg.image_size)
            h = self.encoder(dummy)
            self.enc_shape = h.shape[1:]
            self.enc_flat_dim = int(np.prod(self.enc_shape))

        self.fc_mu = nn.Linear(self.enc_flat_dim, cfg.latent_dim)
        self.fc_logvar = nn.Linear(self.enc_flat_dim, cfg.latent_dim)

        self.fc_decode = nn.Linear(cfg.latent_dim, self.enc_flat_dim)

        # Decoder: 7x7 -> 14x14 -> 28x28
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(c2, c1, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(c1),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(c1, cfg.in_channels, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid(),
        )

    def encode(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        h = self.encoder(x)
        h = h.view(h.size(0), -1)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        """
        TODO:
        Implementar z = mu + sigma * eps, con eps ~ N(0,I).

        Pistas:
        - sigma = exp(0.5 * logvar)
        - eps debe tener la misma forma que sigma
        - usar torch.randn_like(...)
        """
        # ===== TU CÓDIGO AQUÍ =====
        raise NotImplementedError("Implementar reparameterize")

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        h = self.fc_decode(z)
        h = h.view(z.size(0), *self.enc_shape)
        recon = self.decoder(h)
        return recon

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        TODO:
        1. Codificar x -> mu, logvar
        2. Reparametrizar -> z
        3. Decodificar z -> recon
        4. Devolver recon, mu, logvar, z
        """
        # ===== TU CÓDIGO AQUÍ =====
        raise NotImplementedError("Implementar forward")


# 5. Loss del VAE

Implementá:

\[
\text{loss} = \text{reconstruction} + \beta \cdot \text{KL}
\]

Para imágenes en \([0,1]\) usaremos **binary cross-entropy** como loss de reconstrucción.

> Nota: FashionMNIST no es estrictamente binario, pero BCE funciona bien como baseline didáctico. En extensiones podés comparar con MSE.


In [ ]:
def vae_loss(
    recon_x: torch.Tensor,
    x: torch.Tensor,
    mu: torch.Tensor,
    logvar: torch.Tensor,
    beta: float = 1.0,
    free_bits: float = 0.0,
) -> Dict[str, torch.Tensor]:
    """
    Retorna un diccionario con:
    - loss: recon + beta * kl
    - recon: término de reconstrucción promedio por muestra
    - kl: KL promedio por muestra

    TODO:
    1. Calcular BCE entre recon_x y x con reduction='sum'.
    2. Dividir por batch size para reportar promedio por muestra.
    3. Calcular KL cerrada:
       -0.5 * sum(1 + logvar - mu^2 - exp(logvar))
    4. Dividir KL por batch size.
    5. Aplicar beta.
    6. Devolver dict.
    """
    batch_size = x.size(0)

    # ===== TU CÓDIGO AQUÍ =====
    raise NotImplementedError("Implementar vae_loss")


# 6. Checkpoint automático

Ejecutá esta celda antes de entrenar. Si falla, no continúes: el entrenamiento arrastrará errores.


In [ ]:
def run_checkpoint():
    print("── Checkpoint VAE ──")
    model = ConvVAE(cfg).to(device)
    x = torch.rand(8, 1, cfg.image_size, cfg.image_size, device=device)

    recon, mu, logvar, z = model(x)

    assert recon.shape == x.shape, f"recon shape incorrecta: {recon.shape} vs {x.shape}"
    print("  [OK] recon shape")

    assert mu.shape == (8, cfg.latent_dim), f"mu shape incorrecta: {mu.shape}"
    assert logvar.shape == (8, cfg.latent_dim), f"logvar shape incorrecta: {logvar.shape}"
    assert z.shape == (8, cfg.latent_dim), f"z shape incorrecta: {z.shape}"
    print("  [OK] mu/logvar/z shape")

    for name, t in [("recon", recon), ("mu", mu), ("logvar", logvar), ("z", z)]:
        assert torch.isfinite(t).all(), f"{name} contiene NaN o Inf"
    print("  [OK] sin NaN/Inf")

    assert recon.min() >= 0 and recon.max() <= 1, "recon debe estar en [0,1]"
    print("  [OK] recon en [0,1]")

    ld = vae_loss(recon, x, mu, logvar, beta=1.0)
    for k in ["loss", "recon", "kl"]:
        assert k in ld, f"Falta key '{k}' en loss_dict"
        assert torch.isfinite(ld[k]), f"{k} no es finita"
    assert ld["loss"] >= ld["recon"], "loss debería ser recon + beta*kl"
    print(f"  [OK] loss: total={ld['loss'].item():.2f}, recon={ld['recon'].item():.2f}, kl={ld['kl'].item():.2f}")

    mu0 = torch.zeros(4, cfg.latent_dim, device=device)
    logvar0 = torch.zeros(4, cfg.latent_dim, device=device)
    dummy = torch.zeros(4, 1, cfg.image_size, cfg.image_size, device=device)
    kl0 = vae_loss(dummy, dummy, mu0, logvar0, beta=1.0)["kl"]
    assert torch.allclose(kl0, torch.tensor(0.0, device=device), atol=1e-5), \
        f"KL(N(0,I)||N(0,I)) debe ser 0, dio {kl0.item():.6f}"
    print("  [OK] KL(N(0,I)||N(0,I)) = 0")

    # Si reparameterize funciona, z no debería ser idéntico a mu.
    assert not torch.allclose(z, mu), "z == mu: probablemente falta ruido en reparameterize"
    print("  [OK] reparametrización activa")

    print("[CHECKPOINT PASADO]")

run_checkpoint()


# 7. Entrenamiento con KL annealing

El término KL puede dominar muy temprano. Para estabilizar el aprendizaje usaremos **KL annealing**:

\[
\beta_{\text{eff}}(e) = \beta \cdot \min\left(1, \frac{e+1}{E_{\text{anneal}}}\right)
\]

Al principio el modelo prioriza reconstrucción. Gradualmente se fuerza la regularización latente.


In [ ]:
def make_loaders(cfg: VAEConfig):
    num_workers = cfg.num_workers if device == "cuda" else 0
    pin_memory = (device == "cuda")
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=pin_memory)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False,
                            num_workers=num_workers, pin_memory=pin_memory)
    return train_loader, val_loader

def beta_schedule(epoch: int, cfg: VAEConfig) -> float:
    if cfg.kl_anneal_epochs <= 0:
        return cfg.beta
    return cfg.beta * min(1.0, (epoch + 1) / cfg.kl_anneal_epochs)

def run_epoch(model, loader, optimizer=None, cfg: Optional[VAEConfig] = None, epoch: int = 0):
    is_train = optimizer is not None
    model.train(is_train)

    beta_eff = beta_schedule(epoch, cfg) if cfg is not None else 1.0
    free_bits = cfg.free_bits if cfg is not None else 0.0

    totals = {"loss": 0.0, "recon": 0.0, "kl": 0.0}
    n = 0

    with torch.set_grad_enabled(is_train):
        for x, _ in loader:
            x = x.to(device)

            if is_train:
                optimizer.zero_grad(set_to_none=True)

            recon, mu, logvar, z = model(x)
            ld = vae_loss(recon, x, mu, logvar, beta=beta_eff, free_bits=free_bits)
            loss = ld["loss"]

            if is_train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()

            bs = x.size(0)
            for k in totals:
                totals[k] += float(ld[k].detach().cpu()) * bs
            n += bs

    out = {k: v / n for k, v in totals.items()}
    out["beta_eff"] = beta_eff
    return out

def train_vae(cfg: VAEConfig, tag: str = "", verbose: bool = True):
    torch.manual_seed(SEED)
    train_loader, val_loader = make_loaders(cfg)

    model = ConvVAE(cfg).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    history = []

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    if verbose:
        print(f"--- Entrenando {tag or 'VAE'} | params={n_params:,} | epochs={cfg.epochs} ---")

    for epoch in range(cfg.epochs):
        tr = run_epoch(model, train_loader, optimizer=optimizer, cfg=cfg, epoch=epoch)
        with torch.no_grad():
            va = run_epoch(model, val_loader, optimizer=None, cfg=cfg, epoch=epoch)

        row = {
            "epoch": epoch + 1,
            **{f"train_{k}": v for k, v in tr.items()},
            **{f"val_{k}": v for k, v in va.items()},
        }
        history.append(row)

        if verbose:
            print(
                f"ep {epoch+1:02d}/{cfg.epochs} | "
                f"beta_eff={tr['beta_eff']:.2f} | "
                f"train loss={tr['loss']:.2f} rec={tr['recon']:.2f} kl={tr['kl']:.2f} | "
                f"val loss={va['loss']:.2f} rec={va['recon']:.2f} kl={va['kl']:.2f}"
            )

    return model, history


In [ ]:
# Entrenamiento baseline
base_cfg = VAEConfig(beta=1.0, latent_dim=16)
vae_baseline, hist_baseline = train_vae(base_cfg, tag="baseline beta=1 dim=16")


In [ ]:
def plot_history(history, title="Curvas de entrenamiento"):
    epochs = [h["epoch"] for h in history]

    plt.figure(figsize=(8, 4))
    plt.plot(epochs, [h["train_loss"] for h in history], marker="o", label="train total")
    plt.plot(epochs, [h["val_loss"] for h in history], marker="o", label="val total")
    plt.xlabel("Época")
    plt.ylabel("Loss total")
    plt.title(title)
    plt.grid(True, alpha=0.4)
    plt.legend()
    plt.show()

    plt.figure(figsize=(8, 4))
    plt.plot(epochs, [h["val_recon"] for h in history], marker="o", label="val recon")
    plt.plot(epochs, [h["val_kl"] for h in history], marker="o", label="val KL")
    plt.xlabel("Época")
    plt.ylabel("Valor promedio por muestra")
    plt.title("Descomposición de la loss en validación")
    plt.grid(True, alpha=0.4)
    plt.legend()
    plt.show()

plot_history(hist_baseline, title="Baseline VAE")


# 8. Evaluación visual: reconstrucción, generación e interpolación

Evaluaremos tres propiedades distintas:

1. **Reconstrucción:** ¿el VAE conserva la información de entrada?
2. **Muestreo desde prior:** ¿el decoder genera muestras plausibles desde \(z \sim \mathcal{N}(0,I)\)?
3. **Interpolación:** ¿el espacio latente es continuo y semánticamente suave?


In [ ]:
@torch.no_grad()
def show_reconstructions(model, loader, n=16, title=""):
    model.eval()
    x, y = next(iter(loader))
    x = x[:n].to(device)
    recon, mu, logvar, z = model(x)
    pair = torch.cat([x.cpu(), recon.cpu()], dim=0)
    mse = F.mse_loss(recon, x).item()
    show_image_grid(
        pair,
        title=f"{title} | arriba: originales, abajo: reconstrucciones | MSE={mse:.4f}",
        nrow=n,
        figsize=(14, 4)
    )

@torch.no_grad()
def sample_prior(model, n=32, title=""):
    model.eval()
    z = torch.randn(n, model.cfg.latent_dim, device=device)
    samples = model.decode(z)
    show_image_grid(samples, title=f"{title} | muestras desde prior z~N(0,I)", nrow=8, figsize=(10, 5))

@torch.no_grad()
def interpolate_latent(model, loader, idx_a=0, idx_b=5, steps=12, title=""):
    model.eval()
    x, y = next(iter(loader))
    xa = x[idx_a:idx_a+1].to(device)
    xb = x[idx_b:idx_b+1].to(device)

    mu_a, _ = model.encode(xa)
    mu_b, _ = model.encode(xb)

    alphas = torch.linspace(0, 1, steps, device=device).view(-1, 1)
    z_interp = (1 - alphas) * mu_a + alphas * mu_b
    imgs = model.decode(z_interp)

    show_image_grid(imgs, title=f"{title} | interpolación lineal en espacio latente", nrow=steps, figsize=(14, 2))
    print(f"Clase A: {CLASSES[int(y[idx_a])]}  ->  Clase B: {CLASSES[int(y[idx_b])]}")

train_loader_base, val_loader_base = make_loaders(base_cfg)

show_reconstructions(vae_baseline, val_loader_base, n=16, title="Baseline beta=1")
sample_prior(vae_baseline, n=32, title="Baseline beta=1")
interpolate_latent(vae_baseline, val_loader_base, idx_a=0, idx_b=6, steps=12, title="Baseline beta=1")


# 9. Visualización del espacio latente

Proyectamos \(\mu_\phi(x)\) a 2D con PCA.

> Interpretación cuidadosa: que haya clusters por clase sugiere que el VAE aprendió representaciones útiles, pero no prueba por sí solo buena calidad generativa.


In [ ]:
try:
    from sklearn.decomposition import PCA
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False
    print("sklearn no disponible; se saltea PCA.")

@torch.no_grad()
def extract_latents(model, loader, max_batches=50):
    model.eval()
    mus, labels = [], []
    for b, (x, y) in enumerate(loader):
        if b >= max_batches:
            break
        mu, _ = model.encode(x.to(device))
        mus.append(mu.cpu().numpy())
        labels.append(y.numpy())
    return np.concatenate(mus), np.concatenate(labels)

def plot_latent_pca(mus, labels, title="PCA del espacio latente"):
    if not SKLEARN_AVAILABLE:
        return
    proj = PCA(n_components=2, random_state=SEED).fit_transform(mus)

    plt.figure(figsize=(8, 6))
    cmap = plt.cm.get_cmap("tab10", 10)
    for cls in range(10):
        mask = labels == cls
        plt.scatter(proj[mask, 0], proj[mask, 1], s=8, alpha=0.55, color=cmap(cls), label=CLASSES[cls])
    plt.title(title)
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.legend(fontsize=7, markerscale=2)
    plt.grid(True, alpha=0.3)
    plt.show()

mus_base, labels_base = extract_latents(vae_baseline, val_loader_base)
print("Latentes extraídos:", mus_base.shape)
plot_latent_pca(mus_base, labels_base, title="Baseline beta=1 | PCA de mu")


# 10. Ablation study: β-VAE

Ahora compararemos valores de \(\beta\) manteniendo todo lo demás fijo.

\[
\text{loss} = \text{recon} + \beta \cdot KL
\]

Valores sugeridos:

- \(\beta=0.1\): poca regularización. Esperamos mejor reconstrucción, pero peor muestreo desde el prior.
- \(\beta=1.0\): VAE estándar.
- \(\beta=4.0\): mayor presión hacia el prior. Esperamos espacio más regularizado, pero reconstrucción más borrosa.

**Importante:** no basta con mirar la loss total, porque cambia la escala con β. Compará por separado `val_recon`, `val_kl`, muestras e interpolaciones.


In [ ]:
betas_to_test = [0.1, 1.0, 4.0]
epochs_ablation = 8 if device != "cpu" else 3
results_beta = {}

for b in betas_to_test:
    cfg_b = VAEConfig(
        latent_dim=16,
        beta=b,
        epochs=epochs_ablation,
        kl_anneal_epochs=max(1, min(4, epochs_ablation)),
        batch_size=128,
    )
    model_b, hist_b = train_vae(cfg_b, tag=f"beta={b}", verbose=False)
    results_beta[b] = {"model": model_b, "history": hist_b, "cfg": cfg_b}

    last = hist_b[-1]
    n_params = sum(p.numel() for p in model_b.parameters() if p.requires_grad)
    print(
        f"beta={b:>4} | params={n_params:,} | "
        f"val_loss={last['val_loss']:.2f} | val_recon={last['val_recon']:.2f} | val_kl={last['val_kl']:.2f}"
    )


In [ ]:
print(f"{'beta':>6} | {'val_recon':>10} | {'val_KL':>8} | {'val_total':>10} | Comentario")
print("-" * 76)

for b, res in results_beta.items():
    last = res["history"][-1]
    # TODO opcional: reemplazar este comentario automático por tu interpretación.
    comment = "Completar interpretación"
    print(
        f"{b:>6.1f} | {last['val_recon']:>10.2f} | {last['val_kl']:>8.2f} | "
        f"{last['val_loss']:>10.2f} | {comment}"
    )


In [ ]:
# Visualización comparativa por beta
_, val_loader_tmp = make_loaders(base_cfg)

for b in betas_to_test:
    print(f"\n=== beta = {b} ===")
    model_b = results_beta[b]["model"]
    show_reconstructions(model_b, val_loader_tmp, n=12, title=f"beta={b}")
    sample_prior(model_b, n=16, title=f"beta={b}")


# 11. Análisis de posterior collapse

Decimos que hay **posterior collapse** cuando el encoder deja de usar parte o todo el espacio latente:

\[
q_\phi(z|x) \approx p(z)
\]

Indicios:

- KL total muy baja.
- Muchas dimensiones latentes con KL \(\approx 0\).
- Reconstrucciones poco dependientes del input.
- Muestras del prior y reconstrucciones muy parecidas entre sí.

Graficaremos la KL promedio por dimensión.


In [ ]:
@torch.no_grad()
def analyze_collapse(model, loader, tag="", threshold=0.1):
    model.eval()
    kl_dims = []
    for b, (x, _) in enumerate(loader):
        if b >= 20:
            break
        _, mu, logvar, _ = model(x.to(device))
        # KL por dimensión, antes de sumar sobre z.
        kl_raw = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp())
        kl_dims.append(kl_raw.mean(dim=0).detach().cpu())

    kl_mean = torch.stack(kl_dims).mean(dim=0).numpy()
    n_inactive = int((kl_mean < threshold).sum())
    n_active = len(kl_mean) - n_inactive

    print(f"\n{tag}")
    print(f"  KL total aproximada: {kl_mean.sum():.3f}")
    print(f"  Dimensiones activas   (KL >= {threshold}): {n_active}")
    print(f"  Dimensiones inactivas (KL <  {threshold}): {n_inactive}")

    plt.figure(figsize=(9, 3))
    colors = ["tab:red" if k < threshold else "tab:blue" for k in kl_mean]
    plt.bar(range(len(kl_mean)), kl_mean, color=colors)
    plt.axhline(threshold, color="red", linestyle="--", alpha=0.6, label=f"umbral {threshold}")
    plt.xlabel("Dimensión latente")
    plt.ylabel("KL promedio")
    plt.title(f"{tag} | KL por dimensión")
    plt.legend()
    plt.tight_layout()
    plt.show()

for b in betas_to_test:
    analyze_collapse(results_beta[b]["model"], val_loader_tmp, tag=f"beta={b}")


# 12. Extensión opcional A — Olivetti Faces

Esta sección es opcional. Sirve para adaptar el pipeline a imágenes de caras \(64 \times 64\).

**Importante:** no hace falta correrla para aprobar. Si la corrés, discutí también los límites éticos de usar datos faciales: consentimiento, privacidad, sesgos y riesgo de uso indebido.


In [ ]:
RUN_OLIVETTI = False

if RUN_OLIVETTI:
    try:
        from sklearn.datasets import fetch_olivetti_faces
        faces = fetch_olivetti_faces(shuffle=True, random_state=SEED)
        X_faces = torch.tensor(faces.images, dtype=torch.float32).unsqueeze(1)
        y_faces = torch.tensor(faces.target, dtype=torch.long)

        ds_faces = TensorDataset(X_faces, y_faces)
        n_train = int(0.8 * len(ds_faces))
        n_val = len(ds_faces) - n_train
        faces_train, faces_val = torch.utils.data.random_split(
            ds_faces, [n_train, n_val], generator=torch.Generator().manual_seed(SEED)
        )

        print("Olivetti Faces:", X_faces.shape)
        show_image_grid(X_faces[:40], title="Olivetti Faces — muestra", nrow=10, figsize=(10, 5))

        # Desafío:
        # - adaptar ConvVAE a image_size=64
        # - agregar una capa convolucional más
        # - entrenar y comparar reconstrucciones/interpolaciones
    except Exception as e:
        print("No se pudo cargar Olivetti Faces:", e)


# 13. Extensión opcional B — GAN mínimo

Un GAN contiene:

- **Generador \(G\):** transforma ruido \(z\) en una imagen falsa.
- **Discriminador \(D\):** intenta distinguir imágenes reales de generadas.

Objetivo conceptual:

\[
\min_G \max_D
\mathbb{E}_{x \sim p_{data}}[\log D(x)]
+
\mathbb{E}_{z \sim p(z)}[\log(1-D(G(z)))]
\]

La extensión consiste en implementar un GAN MLP mínimo sobre \(28 \times 28\) y compararlo con el VAE.

**No es obligatorio** porque los GANs son más inestables y sensibles a hiperparámetros.


In [ ]:
RUN_GAN = False

if RUN_GAN:
    class MLPGenerator(nn.Module):
        def __init__(self, z_dim=64, img_dim=28*28):
            super().__init__()
            self.z_dim = z_dim
            self.net = nn.Sequential(
                nn.Linear(z_dim, 256), nn.LeakyReLU(0.2, inplace=True),
                nn.Linear(256, 512), nn.LeakyReLU(0.2, inplace=True),
                nn.Linear(512, img_dim), nn.Sigmoid(),
            )

        def forward(self, z):
            return self.net(z).view(z.size(0), 1, 28, 28)

    class MLPDiscriminator(nn.Module):
        def __init__(self, img_dim=28*28):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(img_dim, 512), nn.LeakyReLU(0.2, inplace=True),
                nn.Dropout(0.3),
                nn.Linear(512, 256), nn.LeakyReLU(0.2, inplace=True),
                nn.Dropout(0.3),
                nn.Linear(256, 1),  # logits
            )

        def forward(self, x):
            return self.net(x.view(x.size(0), -1))

    # TODO opcional:
    # 1. Instanciar G y D.
    # 2. Entrenar D con reales=1 y fakes=0.
    # 3. Entrenar G para que D(fake)=1.
    # 4. Graficar muestras y curvas.
    raise NotImplementedError("Extensión opcional GAN")


# 14. Extensión opcional C — Matrices de conectividad cerebral

Esta sección conecta el TP con aplicaciones biomédicas de deep learning.

Una matriz de conectividad funcional puede representarse como:

\[
X \in \mathbb{R}^{N \times C \times R \times R}
\]

donde:

- \(N\): sujetos.
- \(C\): canales de conectividad, por ejemplo Pearson, MI, dFC, Granger.
- \(R\): número de regiones cerebrales.
- \(X[n,c,i,j]\): conectividad entre regiones \(i\) y \(j\) del sujeto \(n\), canal \(c\).

## Advertencia ética y metodológica

No subas datos clínicos identificables. Antes de compartir datasets biomédicos:

- quitar nombres, IDs, fechas exactas y metadatos sensibles;
- verificar consentimiento y aprobación ética;
- evitar imágenes anatómicas que permitan reconstrucción facial;
- documentar preprocesamiento y criterios de exclusión;
- compartir preferentemente datos agregados, anonimizados o sintéticos.

## Formato sugerido si la cátedra provee datos

Archivo `.npz`:

```python
X: float32, shape (N, C, R, R)
y: int64,   shape (N,)    # opcional
channels: lista de nombres de canales
```


In [ ]:
def make_synthetic_connectomes(n=256, channels=3, rois=32, seed=SEED):
    """
    Genera matrices simétricas sintéticas para experimentar sin datos clínicos reales.
    No representa pacientes reales.
    """
    rng = np.random.default_rng(seed)
    X = rng.normal(0, 0.3, size=(n, channels, rois, rois)).astype("float32")
    X = 0.5 * (X + np.transpose(X, (0, 1, 3, 2)))
    for i in range(n):
        for c in range(channels):
            np.fill_diagonal(X[i, c], 0.0)

    # Etiqueta sintética: clase 1 tiene un bloque más conectado.
    y = np.zeros(n, dtype="int64")
    y[n//2:] = 1
    X[n//2:, :, :8, :8] += 0.5
    X[n//2:, :, :8, :8] = 0.5 * (X[n//2:, :, :8, :8] + np.transpose(X[n//2:, :, :8, :8], (0,1,3,2)))
    return X, y

RUN_CONNECTOME_SYNTHETIC = False

if RUN_CONNECTOME_SYNTHETIC:
    X_conn, y_conn = make_synthetic_connectomes()
    print("X_conn:", X_conn.shape, "y:", y_conn.shape)

    # TODO opcional:
    # - adaptar el encoder para in_channels=C e image_size=R
    # - entrenar un VAE sobre matrices
    # - discutir por qué la localidad 2D de una matriz depende del ordenamiento de ROIs


# 15. Informe final dentro del notebook

Completá esta sección antes de entregar.

## 15.1 Implementación

Respondé:

1. ¿Qué diferencia hay entre un autoencoder determinístico y un VAE?
2. ¿Por qué el encoder devuelve `mu` y `logvar`?
3. ¿Qué hace el truco de reparametrización?
4. ¿Por qué podemos calcular la KL en forma cerrada?
5. ¿Qué rol tiene `beta`?

---

## 15.2 Resultados cuantitativos

Completá la tabla con tus resultados.

| Modelo | latent_dim | beta | val_recon | val_KL | val_total | Comentario |
|---|---:|---:|---:|---:|---:|---|
| baseline | 16 | 1.0 |  |  |  |  |
| beta bajo | 16 | 0.1 |  |  |  |  |
| beta alto | 16 | 4.0 |  |  |  |  |

---

## 15.3 Resultados visuales

Comentá:

1. ¿Qué detalles reconstruye bien el modelo?
2. ¿Qué detalles pierde?
3. ¿Las muestras desde el prior parecen imágenes plausibles?
4. ¿Las interpolaciones son suaves?
5. ¿Qué configuración te parece mejor y por qué?

---

## 15.4 Posterior collapse

Respondé:

1. ¿Qué es posterior collapse?
2. ¿Qué evidencias buscaste en tus modelos?
3. ¿Qué relación observaste entre β y KL por dimensión?
4. ¿Alguna dimensión latente quedó inactiva?

---

## 15.5 Comparación VAE vs GAN

Respondé:

1. ¿Por qué un VAE tiende a generar imágenes más borrosas?
2. ¿Por qué un GAN puede producir muestras más nítidas?
3. ¿Por qué los GANs son más difíciles de entrenar?
4. ¿Qué métrica usarías para comparar modelos generativos más seriamente?

---

## 15.6 Conclusión

En 10–15 líneas:

- qué aprendió el VAE;
- qué limitaciones observaste;
- qué configuración funcionó mejor;
- qué cambiarías con más tiempo/GPU/datos;
- qué aprendiste sobre modelos generativos.


# 16. Preguntas sugeridas para la defensa oral

1. Escribí la ELBO y explicá cada término.
2. ¿Por qué un VAE puede generar imágenes nuevas y un autoencoder común no necesariamente?
3. ¿Qué ocurre si \(\beta=0\)?
4. ¿Qué ocurre si \(\beta\) es demasiado grande?
5. ¿Qué es posterior collapse?
6. ¿Por qué usamos `logvar` en lugar de `sigma`?
7. ¿Por qué buena reconstrucción no garantiza buena generación?
8. ¿Qué mirás en una interpolación latente?
9. ¿Qué diferencias hay entre VAE y GAN?
10. ¿Cómo adaptarías este TP a datos biomédicos de conectividad cerebral?


# 17. Rúbrica sugerida

| Ítem | Peso |
|---|---:|
| Implementación correcta de ConvVAE y reparametrización | 20% |
| Implementación correcta de loss VAE + KL | 20% |
| Entrenamiento reproducible con curvas y métricas | 15% |
| Evaluación visual: reconstrucción, prior e interpolación | 15% |
| Ablation β-VAE y análisis de posterior collapse | 15% |
| Informe final y defensa conceptual | 15% |

## Para nota alta

- Comparaciones controladas.
- Interpretación crítica de los resultados.
- Buena conexión entre teoría y código.
- No sobreinterpretar muestras visuales.
- Una extensión opcional bien justificada.


# Referencias

- Kingma, D. P., & Welling, M. (2013). *Auto-Encoding Variational Bayes.*
- Higgins, I. et al. (2017). *β-VAE: Learning Basic Visual Concepts with a Constrained Variational Framework.*
- Goodfellow, I. et al. (2014). *Generative Adversarial Nets.*
- PyTorch documentation.
- Implementaciones didácticas de VAE/GAN en PyTorch usadas como inspiración.
